# ASL Sign Language Classifier
**Forman CS Club · AI Hackathon 2026**

This notebook clones the GitHub repo and trains the model using Kaggle's GPU.

## Step 1: Clone the repo

In [ ]:
# # Replace with your actual GitHub URL
# !git clone https://github.com/YOUR_USERNAME/asl-sign-classifier.git
# %cd asl-sign-classifier
# !pip install -r requirements.txt -q

## Step 2: Verify dataset and GPU

In [ ]:
import os, torch

DATA_DIR = '/kaggle/input/YOUR-DATASET-NAME'  # ← update with actual Kaggle dataset path

print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU only')
print('CUDA available:', torch.cuda.is_available())
print()

train_classes = os.listdir(os.path.join(DATA_DIR, 'train'))
print(f'Train classes ({len(train_classes)}):', sorted(train_classes))

test_images = os.listdir(os.path.join(DATA_DIR, 'test'))
print(f'Test images  : {len(test_images)}')

# Count training images
total = sum(len(os.listdir(os.path.join(DATA_DIR, 'train', c))) for c in train_classes)
print(f'Total train  : {total}')

## Step 3: Train

In [ ]:
!python src/train.py \
    --data_dir  $DATA_DIR \
    --save_dir  /kaggle/working/checkpoints \
    --arch      efficientnet_b3 \
    --epochs    20 \
    --batch_size 64 \
    --lr        3e-4 \
    --workers   2

## Step 4: Generate submission with TTA

In [ ]:
os.makedirs('/kaggle/working/submissions', exist_ok=True)

!python src/inference.py \
    --data_dir  $DATA_DIR \
    --ckpt      /kaggle/working/checkpoints/best_model.pth \
    --output    /kaggle/working/submissions/submission.csv \
    --tta       5 \
    --batch_size 128

## Step 5: Validate submission format

In [ ]:
import pandas as pd

sub = pd.read_csv('/kaggle/working/submissions/submission.csv')
print('Shape:', sub.shape)
print('Columns:', sub.columns.tolist())
print('\nLabel distribution:')
print(sub['label'].value_counts().sort_index())
print('\nFirst 10 rows:')
sub.head(10)

## Step 6 (Optional): Training history plot

In [ ]:
import json, matplotlib.pyplot as plt

with open('/kaggle/working/checkpoints/history.json') as f:
    history = json.load(f)

epochs   = [h['epoch'] for h in history]
tr_accs  = [h['tr_acc'] for h in history]
vl_accs  = [h['vl_acc'] for h in history]

plt.figure(figsize=(10, 4))
plt.plot(epochs, tr_accs, 'b-o', label='Train Acc')
plt.plot(epochs, vl_accs, 'r-o', label='Val Acc')
plt.xlabel('Epoch'); plt.ylabel('Accuracy (%)')
plt.title('Training Progress'); plt.legend(); plt.grid(True)
plt.tight_layout(); plt.savefig('/kaggle/working/training_history.png', dpi=150)
plt.show()
print(f'Best val acc: {max(vl_accs):.2f}%')